# POC: Token Economics Observability with NVIDIA NeMo Agent Toolkit and NeMo Relay

This notebook presents a CFO-facing agent observability prototype for a subscription finance workflow.

The goal is to measure the token economics of an entire agentic pipeline:

1. Agent planning and routing.
2. Retrieval and tool calls.
3. CFO analysis and summary generation.
4. Validation or judge steps.
5. Retries, cached context, and handoffs.
6. Subprocess or external-agent spans brought in through NeMo Relay.

The notebook has two layers:

- A runnable local economics demo using simulated NAT and Relay-style telemetry.
- A deployment blueprint for wiring the same metrics to NVIDIA NeMo Agent Toolkit telemetry and NeMo Relay ATOF events.

> Safety: `RUN_COMMANDS = True` runs the local economics demo. Live NVIDIA runtime and infrastructure commands are printed unless `RUN_LIVE_NVIDIA_COMMANDS` or `RUN_INFRA_COMMANDS` is explicitly enabled.

## Why This Is Worth Presenting

This is a strong POC because the value is business-readable:

| Question | Metric |
|---|---|
| What does one CFO analysis cost? | Cost per workflow run |
| Which step burns tokens? | Cost and token share by stage |
| Are retries expensive? | Retry token waste |
| Is retrieval helping or bloating context? | Context amplification ratio |
| Which model mix is economical? | Cost by model and scenario |
| Can non-NAT work be included? | Relay ATOF spans folded into the same trace |

The story is not just "observability." It is "unit economics for agentic AI."

## Reference Architecture

```mermaid
flowchart LR
    U["Finance user"] --> NAT["NeMo Agent Toolkit workflow"]
    NAT --> P["Planner / router LLM"]
    NAT --> T["Finance tools"]
    NAT --> R["Retrieval / policy context"]
    NAT --> S["CFO summary LLM"]
    NAT --> J["Validator / judge LLM"]
    EXT["External subprocess or tool agent"] --> RELAY["NeMo Relay ATOF events"]
    RELAY --> BRIDGE["Relay telemetry bridge"]
    BRIDGE --> NATTRACE["NAT intermediate-step stream"]
    NAT --> NATTRACE
    NATTRACE --> OTEL["OTel / Phoenix / file exporter"]
    OTEL --> ECON["Token economics notebook"]
    ECON --> CFO["Cost, latency, and optimization readout"]
```

NAT provides the workflow and telemetry backbone. NeMo Relay brings external or subprocess telemetry into the same trace stream. The economics layer turns token usage into CFO-readable cost and optimization levers.

## Current NVIDIA Building Blocks

Relevant pieces in the current NVIDIA documentation:

- NeMo Agent Toolkit supports workflow construction, LLMs, tools, telemetry exporters, profiling, evaluation, and `nat run` / `nat eval`.
- NAT observability can send traces to Phoenix, OpenTelemetry collectors, Weave, file output, or custom exporters.
- NAT token usage includes prompt, completion, cached, reasoning, and total token counters.
- NeMo Relay emits Agent Trajectory Observability Format, or ATOF, lifecycle events.
- `nat.experimental.relay_telemetry_bridge` maps Relay ATOF events into NAT `IntermediateStep` objects so Relay-instrumented subprocesses can appear in the same NAT trace.

References:

- NAT overview: https://docs.nvidia.com/nemo/agent-toolkit/latest/index.html
- Observe workflows: https://docs.nvidia.com/nemo/agent-toolkit/latest/run-workflows/observe/observe.html
- Profiling: https://docs.nvidia.com/nemo/agent-toolkit/latest/improve-workflows/profiler.html
- Token usage model: https://docs.nvidia.com/nemo/agent-toolkit/latest/api/nat/data_models/token_usage/index.html
- Relay telemetry bridge: https://docs.nvidia.com/nemo/agent-toolkit/latest/api/nat/experimental/relay_telemetry_bridge/index.html

## 0. Configure The POC

Replace the demo prices with approved pricing for the target deployment. The demo numbers are intentionally placeholders.

In [ ]:
from pathlib import Path
import csv
import json
import math
import os
import shlex
import statistics
import subprocess
import textwrap
from collections import defaultdict
from IPython.display import Markdown, display


RUN_COMMANDS = True
RUN_LIVE_NVIDIA_COMMANDS = False
RUN_INFRA_COMMANDS = False


WORK_DIR = Path("/workspace/nat_relay_token_economics_poc")
LOCAL_DEMO_DIR = Path("sample_data")
TRACE_JSONL = LOCAL_DEMO_DIR / "agent_trace_events.jsonl"
ECONOMICS_CSV = LOCAL_DEMO_DIR / "token_economics_summary.csv"


NAT_PROJECT = "cfo_token_economics"
NAT_CONFIG = WORK_DIR / "workflow.yml"
OTEL_CONFIG = WORK_DIR / "otelcollectorconfig.yaml"
ATOF_JSONL = WORK_DIR / "relay_atof_events.jsonl"


# Demo-only price book. Replace with the approved commercial or deployment rate card.
# Values are USD per 1M tokens.
PRICE_BOOK = {
    "nvidia/nemotron-3-nano-30b-a3b": {
        "prompt_per_million": 0.25,
        "completion_per_million": 0.75,
        "reasoning_per_million": 0.75,
        "cached_prompt_per_million": 0.05,
    },
    "nvidia/nemotron-3-super-120b-a12b": {
        "prompt_per_million": 0.80,
        "completion_per_million": 2.40,
        "reasoning_per_million": 2.40,
        "cached_prompt_per_million": 0.16,
    },
    "embedding-model": {
        "prompt_per_million": 0.02,
        "completion_per_million": 0.00,
        "reasoning_per_million": 0.00,
        "cached_prompt_per_million": 0.00,
    },
}


def q(value):
    return shlex.quote(str(value))


def show_command(title, cmd, cwd=None, run=None):
    cmd = textwrap.dedent(cmd).strip()
    should_run = RUN_COMMANDS if run is None else run
    display(Markdown(f"### {title}"))
    print(cmd)
    if should_run:
        subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=True)


display(Markdown(f'''
### POC Profile

| Item | Value |
|---|---|
| Project | `{NAT_PROJECT}` |
| Local demo dir | `{LOCAL_DEMO_DIR}` |
| Trace file | `{TRACE_JSONL}` |
| Local demo commands enabled | `{RUN_COMMANDS}` |
| Live NVIDIA runtime commands enabled | `{RUN_LIVE_NVIDIA_COMMANDS}` |
| Infrastructure commands enabled | `{RUN_INFRA_COMMANDS}` |
| Price book | Demo placeholders, replace before financial readout |
'''))

## 1. Optional: Install And Run The NAT Baseline

Use this section when running in a Python environment with internet access and an NVIDIA API key. The local economics demo later in the notebook does not require these packages. These commands print by default and only run when `RUN_LIVE_NVIDIA_COMMANDS = True`.

In [ ]:
show_command(
    "Install NeMo Agent Toolkit with observability extras",
    '''
    python -m venv .venv
    . .venv/bin/activate
    python -m pip install --upgrade pip
    pip install "nvidia-nat[opentelemetry,phoenix]"
    ''',
    run=RUN_LIVE_NVIDIA_COMMANDS,
)

show_command(
    "Check NAT CLI",
    '''
    . .venv/bin/activate
    nat --help
    nat info components -t tracing
    nat info components -t logging
    ''',
    run=RUN_LIVE_NVIDIA_COMMANDS,
)

## 2. NAT Workflow Skeleton For A CFO Cost Agent

This YAML is the shape to present. Register the finance functions as NAT tools, then turn on telemetry.

For a first live demo, keep the tool implementations tiny and deterministic:

- `invoice_metrics_lookup`: returns sample ARR, NRR, churn, DSO, and invoice counts.
- `revenue_policy_lookup`: returns short policy snippets.
- `variance_calculator`: computes month-over-month changes.
- `cfo_summary_writer`: writes an executive summary.

The core observability point is that every LLM and function step emits trace events.

In [ ]:
workflow_yml = f'''
general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO
      file:
        _type: file
        path: ./nat_token_economics.log
        level: DEBUG
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: {NAT_PROJECT}
      otelcollector:
        _type: otelcollector
        endpoint: http://localhost:4318/v1/traces
        project: {NAT_PROJECT}
      file_backup:
        _type: file

llms:
  planner_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    temperature: 0.0
    chat_template_kwargs:
      enable_thinking: false
  analyst_llm:
    _type: nim
    model_name: nvidia/nemotron-3-super-120b-a12b
    temperature: 0.1
    chat_template_kwargs:
      enable_thinking: false

functions:
  invoice_metrics_lookup:
    _type: python_function
    module: scripts.finance_tools
    function: invoice_metrics_lookup
  revenue_policy_lookup:
    _type: python_function
    module: scripts.finance_tools
    function: revenue_policy_lookup
  variance_calculator:
    _type: python_function
    module: scripts.finance_tools
    function: variance_calculator

workflow:
  _type: react_agent
  llm_name: planner_llm
  tool_names:
    - invoice_metrics_lookup
    - revenue_policy_lookup
    - variance_calculator
  verbose: true
  parse_agent_response_max_retries: 2
'''

print(workflow_yml)

In [ ]:
show_command(
    "Run NAT workflow with observability",
    f'''
    export NVIDIA_API_KEY=<set_me>
    nat run --config_file {q(str(NAT_CONFIG))} --input "Explain what changed in invoicing cost, NRR, and collections this month. Include CFO-ready risks and actions."
    ''',
    run=RUN_LIVE_NVIDIA_COMMANDS,
)

## 3. OpenTelemetry And Phoenix Options

Use Phoenix for a visual trace demo. Use OTel when the goal is to feed a central observability backend or a cost warehouse.

In [ ]:
otelcollector_yaml = '''
receivers:
  otlp:
    protocols:
      http:
        endpoint: 0.0.0.0:4318

processors:
  batch:
    send_batch_size: 100
    timeout: 10s

exporters:
  file:
    path: /otellogs/llm_spans.json
    format: json

service:
  pipelines:
    traces:
      receivers: [otlp]
      processors: [batch]
      exporters: [file]
'''

print(otelcollector_yaml)

show_command(
    "Run Phoenix and OTel collector",
    '''
    docker run --rm -p 4317:4317 -p 6006:6006 arizephoenix/phoenix:13.22
    docker run --rm -p 4318:4318 -v $(pwd)/otelcollectorconfig.yaml:/etc/otelcol-contrib/config.yaml -v $(pwd)/otellogs:/otellogs otel/opentelemetry-collector-contrib:0.128.0
    ''',
    run=RUN_INFRA_COMMANDS,
)

## 4. NeMo Relay Integration Point

Use NeMo Relay when some of the work happens outside the main NAT workflow, for example:

- A subprocess that runs a separate agent.
- A workflow launched by a tool call.
- A non-NAT service that emits ATOF lifecycle events.

The NAT bridge can load Relay ATOF JSONL and convert it into NAT intermediate steps. This is the link that lets the economics report include the entire agentic path rather than only the top-level NAT calls.

In [ ]:
show_command(
    "Bridge Relay ATOF events into NAT trace stream",
    f'''
    python - <<'PY'
    from nat.experimental.relay_telemetry_bridge import load_atof_jsonl
    from nat.experimental.relay_telemetry_bridge import relay_events_to_intermediate_steps

    events = load_atof_jsonl({q(str(ATOF_JSONL))})
    steps = relay_events_to_intermediate_steps(events, root_parent_id="active_nat_span_id")
    print(f"Loaded {{len(events)}} Relay events")
    print(f"Converted to {{len(steps)}} NAT intermediate steps")
    PY
    ''',
    run=RUN_LIVE_NVIDIA_COMMANDS,
)

## 5. Local Demo: Simulated End-To-End Trace

The next cells run without NAT, NeMo Relay, NIM, or a GPU. They generate representative trace events and calculate token economics.

Use this for the first presentation, then replace the simulated events with `standardized_data_all.csv`, OTel JSON, Phoenix export, or Relay ATOF-derived events from the live run.

In [ ]:
sample_events = [
    {
        "trace_id": "run-001",
        "source": "nat",
        "stage": "planner",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-nano-30b-a3b",
        "prompt_tokens": 1380,
        "completion_tokens": 180,
        "reasoning_tokens": 0,
        "cached_tokens": 0,
        "latency_ms": 820,
        "success": True,
    },
    {
        "trace_id": "run-001",
        "source": "nat",
        "stage": "invoice_metrics_lookup",
        "event_type": "TOOL_END",
        "model": None,
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "reasoning_tokens": 0,
        "cached_tokens": 0,
        "latency_ms": 120,
        "success": True,
    },
    {
        "trace_id": "run-001",
        "source": "relay",
        "stage": "external_policy_agent",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-nano-30b-a3b",
        "prompt_tokens": 2450,
        "completion_tokens": 260,
        "reasoning_tokens": 0,
        "cached_tokens": 600,
        "latency_ms": 1040,
        "success": True,
    },
    {
        "trace_id": "run-001",
        "source": "nat",
        "stage": "variance_analysis",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "prompt_tokens": 3120,
        "completion_tokens": 520,
        "reasoning_tokens": 120,
        "cached_tokens": 0,
        "latency_ms": 2100,
        "success": True,
    },
    {
        "trace_id": "run-001",
        "source": "nat",
        "stage": "cfo_summary",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "prompt_tokens": 1960,
        "completion_tokens": 640,
        "reasoning_tokens": 80,
        "cached_tokens": 0,
        "latency_ms": 1880,
        "success": True,
    },
    {
        "trace_id": "run-001",
        "source": "nat",
        "stage": "validator",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-nano-30b-a3b",
        "prompt_tokens": 1540,
        "completion_tokens": 120,
        "reasoning_tokens": 0,
        "cached_tokens": 0,
        "latency_ms": 760,
        "success": True,
    },
    {
        "trace_id": "run-002",
        "source": "nat",
        "stage": "planner",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-nano-30b-a3b",
        "prompt_tokens": 1420,
        "completion_tokens": 220,
        "reasoning_tokens": 0,
        "cached_tokens": 0,
        "latency_ms": 900,
        "success": True,
    },
    {
        "trace_id": "run-002",
        "source": "relay",
        "stage": "external_policy_agent_retry",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-nano-30b-a3b",
        "prompt_tokens": 2600,
        "completion_tokens": 90,
        "reasoning_tokens": 0,
        "cached_tokens": 0,
        "latency_ms": 980,
        "success": False,
        "retry": True,
    },
    {
        "trace_id": "run-002",
        "source": "relay",
        "stage": "external_policy_agent",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-nano-30b-a3b",
        "prompt_tokens": 2300,
        "completion_tokens": 240,
        "reasoning_tokens": 0,
        "cached_tokens": 400,
        "latency_ms": 1020,
        "success": True,
    },
    {
        "trace_id": "run-002",
        "source": "nat",
        "stage": "variance_analysis",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "prompt_tokens": 3350,
        "completion_tokens": 560,
        "reasoning_tokens": 160,
        "cached_tokens": 0,
        "latency_ms": 2220,
        "success": True,
    },
    {
        "trace_id": "run-002",
        "source": "nat",
        "stage": "cfo_summary",
        "event_type": "LLM_END",
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "prompt_tokens": 2060,
        "completion_tokens": 680,
        "reasoning_tokens": 90,
        "cached_tokens": 0,
        "latency_ms": 1930,
        "success": True,
    },
]

LOCAL_DEMO_DIR.mkdir(parents=True, exist_ok=True)
with TRACE_JSONL.open("w", encoding="utf-8") as f:
    for row in sample_events:
        f.write(json.dumps(row) + "\n")

print(f"Wrote {len(sample_events)} simulated trace events to {TRACE_JSONL}")

## 6. Cost Model

The cost model uses reported tokens, not character-count estimates:

```text
cost =
  billable_prompt_tokens * prompt_rate
+ completion_tokens * completion_rate
+ reasoning_tokens * reasoning_rate
+ cached_tokens * cached_prompt_rate
```

If a provider bills cached tokens differently, update `PRICE_BOOK`.

In [ ]:
def token_cost_usd(event, price_book):
    model = event.get("model")
    if not model:
        return 0.0
    rates = price_book.get(model)
    if rates is None:
        raise KeyError(f"Missing price for model: {model}")

    prompt_tokens = int(event.get("prompt_tokens") or 0)
    completion_tokens = int(event.get("completion_tokens") or 0)
    reasoning_tokens = int(event.get("reasoning_tokens") or 0)
    cached_tokens = int(event.get("cached_tokens") or 0)
    billable_prompt_tokens = max(prompt_tokens - cached_tokens, 0)

    return (
        billable_prompt_tokens * rates["prompt_per_million"]
        + completion_tokens * rates["completion_per_million"]
        + reasoning_tokens * rates["reasoning_per_million"]
        + cached_tokens * rates["cached_prompt_per_million"]
    ) / 1_000_000


def enrich(events):
    enriched = []
    for event in events:
        row = dict(event)
        row["total_tokens"] = (
            int(row.get("prompt_tokens") or 0)
            + int(row.get("completion_tokens") or 0)
            + int(row.get("reasoning_tokens") or 0)
        )
        row["cost_usd"] = token_cost_usd(row, PRICE_BOOK)
        enriched.append(row)
    return enriched


events = [json.loads(line) for line in TRACE_JSONL.read_text().splitlines()]
events = enrich(events)

for row in events[:3]:
    print(json.dumps(row, indent=2))

## 7. Executive Summary Metrics

These are the numbers to lead with.

In [ ]:
def group_sum(events, key, value):
    out = defaultdict(float)
    for row in events:
        out[row.get(key) or "none"] += float(row.get(value) or 0)
    return dict(sorted(out.items(), key=lambda kv: kv[1], reverse=True))


def workflow_success(events):
    success_by_trace = defaultdict(lambda: True)
    for row in events:
        success_by_trace[row["trace_id"]]
        if row.get("success") is False and not row.get("retry"):
            success_by_trace[row["trace_id"]] = False
    return success_by_trace


trace_cost = group_sum(events, "trace_id", "cost_usd")
trace_tokens = group_sum(events, "trace_id", "total_tokens")
trace_latency = group_sum(events, "trace_id", "latency_ms")
stage_cost = group_sum(events, "stage", "cost_usd")
model_cost = group_sum(events, "model", "cost_usd")
source_cost = group_sum(events, "source", "cost_usd")

successful_runs = sum(1 for ok in workflow_success(events).values() if ok)
total_cost = sum(row["cost_usd"] for row in events)
total_tokens = sum(row["total_tokens"] for row in events)
retry_cost = sum(row["cost_usd"] for row in events if row.get("retry"))
final_answer_tokens = sum(row["completion_tokens"] for row in events if row["stage"] == "cfo_summary")
amplification_ratio = total_tokens / final_answer_tokens if final_answer_tokens else math.nan

summary = {
    "workflow_runs": len(trace_cost),
    "successful_runs": successful_runs,
    "total_tokens": total_tokens,
    "total_cost_usd": round(total_cost, 6),
    "avg_cost_per_run_usd": round(total_cost / len(trace_cost), 6),
    "avg_cost_per_success_usd": round(total_cost / successful_runs, 6),
    "retry_cost_usd": round(retry_cost, 6),
    "token_amplification_ratio": round(amplification_ratio, 2),
    "p50_latency_ms": round(statistics.median(trace_latency.values()), 1),
}

display(Markdown("### Executive Metrics"))
for key, value in summary.items():
    print(f"{key}: {value}")

## 8. Where The Money Goes

Stage-level and model-level views identify optimization levers:

- Planner too large: move planning to a smaller model.
- Context too large: trim retrieved evidence before summarization.
- Validator too costly: validate only high-risk cases.
- Retry waste high: fix tool schema, parsing, or routing.
- Relay share high: bring external subprocesses under the same optimization discipline.

In [ ]:
def print_table(title, rows):
    display(Markdown(f"### {title}"))
    total = sum(value for _, value in rows) or 1.0
    print(f"{'name':36} {'cost_usd':>12} {'share':>10}")
    print("-" * 62)
    for name, value in rows:
        print(f"{str(name):36} {value:12.6f} {value / total:9.1%}")


print_table("Cost By Stage", list(stage_cost.items()))
print()
print_table("Cost By Model", list(model_cost.items()))
print()
print_table("Cost By Source", list(source_cost.items()))

## 9. Monthly Spend Scenarios

The CFO-facing question is simple: what happens when this goes from demo to production volume?

In [ ]:
monthly_volumes = [1_000, 10_000, 50_000, 100_000]
avg_cost = total_cost / len(trace_cost)

display(Markdown("### Spend Projection"))
print(f"{'monthly_runs':>14} {'projected_llm_cost_usd':>24}")
print("-" * 42)
for volume in monthly_volumes:
    print(f"{volume:14,d} {avg_cost * volume:24.2f}")

## 10. Optimization Scenario

This sensitivity test asks: what if the expensive analysis and summary calls move from the large model to the smaller model after prompt tuning or tool improvements?

In [ ]:
optimized_events = []
for row in events:
    new_row = dict(row)
    if new_row["stage"] in {"variance_analysis", "cfo_summary"}:
        new_row["model"] = "nvidia/nemotron-3-nano-30b-a3b"
    optimized_events.append(new_row)
optimized_events = enrich(optimized_events)

optimized_total = sum(row["cost_usd"] for row in optimized_events)
savings = total_cost - optimized_total
savings_pct = savings / total_cost if total_cost else 0.0

print(f"Baseline cost per run:  ${total_cost / len(trace_cost):.6f}")
print(f"Optimized cost per run: ${optimized_total / len(trace_cost):.6f}")
print(f"Estimated savings:      {savings_pct:.1%}")

## 11. Export The Economics Dataset

This CSV can feed a spreadsheet, dashboard, or BI tool.

In [ ]:
fieldnames = [
    "trace_id",
    "source",
    "stage",
    "event_type",
    "model",
    "prompt_tokens",
    "completion_tokens",
    "reasoning_tokens",
    "cached_tokens",
    "total_tokens",
    "latency_ms",
    "success",
    "retry",
    "cost_usd",
]

with ECONOMICS_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in events:
        writer.writerow({key: row.get(key, "") for key in fieldnames})

print(f"Wrote {ECONOMICS_CSV}")

## 12. Presentation Flow

Use this order:

1. Show the architecture: NAT is the agent framework; Relay brings non-NAT spans into the same trace.
2. Show a single trace: planner, tools, retrieval, analysis, summary, validator.
3. Show token economics: cost per run, cost per successful run, stage cost, retry waste.
4. Show optimization levers: model routing, context trimming, retry reduction, caching.
5. Show scale projection: monthly cost at 1k, 10k, 50k, and 100k CFO analyses.
6. Close with the production path: NAT telemetry exporter to OTel/Phoenix, Relay ATOF bridge, BI dashboard.

## 13. Production Gate

Before using the numbers as financial evidence:

- Replace the demo price book with approved rates.
- Use live token counts from NAT or the inference provider.
- Confirm how cached and reasoning tokens are billed.
- Run at least 50 representative CFO prompts.
- Segment by workflow type, not just aggregate averages.
- Track quality with evals so cost reduction does not hide answer degradation.